# Fraud Detection – Data Preprocessing

Cleans data, engineers features, imputes missing values, encodes categoricals, splits, and **scales all columns** (including label-encoded categoricals) so downstream linear models and PCA work correctly.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib


## 2. Load & Merge Data

In [ ]:
train_trans = pd.read_csv('../data/raw/train_transaction.csv')
train_id    = pd.read_csv('../data/raw/train_identity.csv')

df = train_trans.merge(train_id, on='TransactionID', how='left')
print('Merged Shape:', df.shape)


## 3. Drop Highly Missing Features (>90%)

In [ ]:
missing_pct  = df.isnull().mean() * 100
cols_to_drop = missing_pct[missing_pct > 90].index
df.drop(columns=cols_to_drop, inplace=True)
print(f'Dropped {len(cols_to_drop)} columns with >90% missing values.')
print('Shape after dropping:', df.shape)


## 4. Feature Engineering

In [ ]:
# Log-transform to reduce skew in TransactionAmt
df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])

# Decimal part may carry fraud signal (e.g. rounded vs precise amounts)
df['TransactionAmt_decimal'] = (df['TransactionAmt'] - df['TransactionAmt'].astype(int)) * 1000
print('Engineered features added: TransactionAmt_log, TransactionAmt_decimal')


## 5. Separate Features and Target

In [ ]:
target = 'isFraud'
X = df.drop(columns=[target, 'TransactionID'])
y = df[target]
print('X shape:', X.shape)
print('y distribution:\n', y.value_counts(normalize=True).round(4))


## 6. Handle Missing Values

In [ ]:
# Identify column types BEFORE encoding
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object']).columns.tolist()

# Impute numerical with median, categorical with 'Missing'
X[num_cols] = X[num_cols].fillna(X[num_cols].median())
X[cat_cols] = X[cat_cols].fillna('Missing')
print(f'Imputed {len(num_cols)} numerical and {len(cat_cols)} categorical columns.')
print(f'Remaining missing values: {X.isnull().sum().sum()}')


## 7. Encode Categorical Features

In [ ]:
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
print(f'Label-encoded {len(cat_cols)} categorical columns.')
# After encoding, ALL columns are now numeric (int64 / float64)


## 8. Train-Test Split (Stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Reset index so parquet files have clean 0-based integer index
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)
print('Train shape:', X_train.shape)
print('Test shape: ', X_test.shape)
print('Train fraud rate: {:.3%}'.format(y_train.mean()))
print('Test  fraud rate: {:.3%}'.format(y_test.mean()))


## 9. Scale ALL Features

In [ ]:
all_cols = X_train.columns.tolist()

scaler = StandardScaler()
X_train[all_cols] = scaler.fit_transform(X_train[all_cols])
X_test[all_cols]  = scaler.transform(X_test[all_cols])
print(f'StandardScaler applied to all {len(all_cols)} columns.')
print(f'X_train mean (sample): {X_train.iloc[:, :5].mean().round(4).values}')  # should be ~0
print(f'X_train std  (sample): {X_train.iloc[:, :5].std().round(4).values}')   # should be ~1


## 10. Save Processed Data

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

X_train.to_parquet('../data/processed/X_train.parquet', index=False)
X_test.to_parquet( '../data/processed/X_test.parquet',  index=False)
pd.DataFrame({'isFraud': y_train}).to_parquet('../data/processed/y_train.parquet', index=False)
pd.DataFrame({'isFraud': y_test}).to_parquet( '../data/processed/y_test.parquet',  index=False)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv( '../data/processed/X_test.csv',  index=False)
pd.DataFrame({'isFraud': y_train}).to_csv('../data/processed/y_train.csv', index=False)
pd.DataFrame({'isFraud': y_test}).to_csv( '../data/processed/y_test.csv',  index=False)
print('Processed datasets saved!')


## 11. Save Scaler & Feature Metadata

In [ ]:
joblib.dump(scaler, '../data/processed/scaler.pkl')
print('Scaler saved (fitted on all columns after encoding).')

feature_metadata = {
    'num_cols': num_cols,          # originally-numeric columns
    'cat_cols': cat_cols,          # originally-categorical (now label-encoded)
    'all_cols': all_cols           # all columns (all scaled)
}
joblib.dump(feature_metadata, '../data/processed/feature_metadata.pkl')
print('Feature metadata saved.')


## Summary

| Step | Action |
|---|---|
| Merge | Transaction + Identity on TransactionID |
| Clean | Dropped 12 columns with >90% missing |
| Engineer | TransactionAmt_log, TransactionAmt_decimal |
| Impute | Median (numeric) / 'Missing' (categorical) |
| Encode | LabelEncoder for all categorical columns |
| Split | Stratified 80/20 (random_state=42), reset index |
| **Scale** | **StandardScaler on ALL columns (including encoded categoricals)** |

**Key fix vs original:** previously only originally-numeric columns were scaled;
label-encoded columns had arbitrary integer ranges that corrupted linear models and PCA.
Dataset is now ready for Feature Reduction.